# Tree Search in Python — From Zero to Interview-Ready
### Bloomberg ML Research Engineer preparation | Learn by tracing, coding, and testing

**Learning goals:** Understand tree structure, recursion, breadth-first search (BFS), depth-first search (DFS), text-document search in a hierarchy, and the differences between a tree, graph, and trie. Complete self-checks, coding exercises, and interview questions.

**How to use:** Read a short explanation → predict the result **before** running the code → change the example → attempt the exercise → reveal the solution lower down. Uses Python's standard library only (plus optional `matplotlib` for diagrams). Estimated study order: sections 1–8, then challenges, then mock interview.

> The examples use *synthetic* company disclosures, research notes and ESG labels. These are programming exercises, not investment advice.

## 1. Mental model: what is a tree?
A **tree** is a collection of nodes linked by parent → child relationships. Unlike a general graph, a rooted tree has one **root**, and every other node has **exactly one parent**. There is exactly one path from the root to any node.

```text
                         Documents             ← root, depth 0
                      /             \
                 Research           Disclosures ← depth 1
                /       \          /      \
            Equities   Rates     Climate    Governance ← depth 2
            /     \
         Apple   Nvidia                       ← depth 3
```

Vocabulary: **node** = item; **edge** = connection; **root** = top node; **child/parent** = one level below/above; **leaf** = no children; **depth** = number of edges from root; **height** = longest downward path to a leaf; **subtree** = node together with descendants.

**Tree search** means visiting nodes in some order until a target is found (or we prove none exists). The order matters for performance and what result is found first.

In [1]:
# Our running example: keys are parent nodes, values are their children.
tree = {
    "Documents": ["Research", "Disclosures"],
    "Research": ["Equities", "Rates"],
    "Disclosures": ["Climate", "Governance"],
    "Equities": ["Apple", "Nvidia"],
    "Rates": [], "Climate": [], "Governance": [],
    "Apple": [], "Nvidia": []
}
root = "Documents"
for parent, children in tree.items():
    print(f"{parent:12} -> {children}")

Documents    -> ['Research', 'Disclosures']
Research     -> ['Equities', 'Rates']
Disclosures  -> ['Climate', 'Governance']
Equities     -> ['Apple', 'Nvidia']
Rates        -> []
Climate      -> []
Governance   -> []
Apple        -> []
Nvidia       -> []


### Check your understanding (write an answer first)
1. What is the root? What is the depth of `Nvidia`?
2. Name three leaves.
3. Is `Research` a leaf? Why or why not?
4. How many edges are on the path from `Documents` to `Nvidia`?

<details><summary>Reveal answers</summary>

1. `Documents`; depth 3. 2. For example `Apple`, `Nvidia`, `Rates`. 3. No, it has children. 4. Three edges.

</details>

## 2. BFS: breadth-first search (level by level)
Think of searching a building **floor by floor**. BFS visits all nodes at depth 0, then depth 1, then depth 2, etc. It uses a **queue** (first in, first out, FIFO): `collections.deque` and `popleft()`.

```text
Visit order: Documents → Research → Disclosures → Equities → Rates
             → Climate → Governance → Apple → Nvidia
```

BFS is excellent when we want the **nearest match by number of edges** in an unweighted tree or graph. It often holds an entire level in memory.

In [4]:
tree

{'Documents': ['Research', 'Disclosures'],
 'Research': ['Equities', 'Rates'],
 'Disclosures': ['Climate', 'Governance'],
 'Equities': ['Apple', 'Nvidia'],
 'Rates': [],
 'Climate': [],
 'Governance': [],
 'Apple': [],
 'Nvidia': []}

In [3]:
from collections import deque

def bfs(tree, start, target):
    queue = deque([start])
    visited_order = []
    while queue:
        node = queue.popleft()            # Take the oldest entry
        visited_order.append(node)
        if node == target:
            return True, visited_order
        queue.extend(tree.get(node, []))  # Add children to the back
    return False, visited_order

found, visited = bfs(tree, "Documents", "Nvidia")
print("Found:", found)
print("Visit order:", " → ".join(visited))

Found: True
Visit order: Documents → Research → Disclosures → Equities → Rates → Climate → Governance → Apple → Nvidia


### Trace the BFS queue (the most important intuition)
At each iteration: **remove from LEFT**, test, then **append children on RIGHT**.

| Action | Queue (front on left) |
|---|---|
| Start | `[Documents]` |
| Visit Documents | `[Research, Disclosures]` |
| Visit Research | `[Disclosures, Equities, Rates]` |
| Visit Disclosures | `[Equities, Rates, Climate, Governance]` |
| Visit Equities | `[Rates, Climate, Governance, Apple, Nvidia]` |

**Prediction:** When BFS finds `Climate`, does it inspect `Apple` first? Run the next cell.

In [ ]:
for target in ["Climate", "Apple", "NotHere"]:
    found, order = bfs(tree, root, target)
    print(target, "->", found, "; visited:", order)

### Why `deque`, not a regular list?
`deque.popleft()` is **O(1)**. `list.pop(0)` must shift elements, taking **O(n)** per pop. `list.append()` and `deque.append()` are both O(1) amortized. For a tree with **N nodes**, BFS takes **O(N)** time in the worst case; its queue can grow to **O(W)** space, where **W** is the maximum number of queued nodes (at most O(N)).

## 3. DFS: depth-first search (go deep, then backtrack)
Imagine exploring a folder, opening the first subfolder, then its first subfolder, until you cannot go further. Then **backtrack** and explore siblings.

**Preorder DFS** with child order as listed:
`Documents → Research → Equities → Apple → Nvidia → Rates → Disclosures → Climate → Governance`

DFS uses a **stack** (last in, first out, LIFO), either explicit or via the Python call stack (recursion). DFS does **not** guarantee the fewest edges to the first found match.

In [ ]:
def dfs_recursive(tree, node, target, order=None):
    if order is None:
        order = []
    order.append(node)
    if node == target:
        return True, order
    for child in tree.get(node, []):
        found, order = dfs_recursive(tree, child, target, order)
        if found:
            return True, order
    return False, order

print(dfs_recursive(tree, root, "Nvidia"))
print(dfs_recursive(tree, root, "NotHere"))

### Understanding recursion, not memorising it
A recursive DFS call has **two pieces**:
1. **Base case:** If this node matches the target, stop.
2. **Recursive step:** For each child, search its subtree; if not found, return and try the next child.

**Call-stack trace for `Apple`:** `dfs(Documents)` → `dfs(Research)` → `dfs(Equities)` → `dfs(Apple)` → found. Returning to callers is **unwinding the stack**.

A subtle point: `order=None` is used instead of `order=[]` in the function signature. A mutable default list would persist between independent calls.

In [ ]:
def dfs_iterative(tree, start, target):
    stack = [start]
    visit_order = []
    while stack:
        node = stack.pop()             # Take the newest entry
        visit_order.append(node)
        if node == target:
            return True, visit_order
        # Reverse ensures the LEFTMOST child is processed first.
        stack.extend(reversed(tree.get(node, [])))
    return False, visit_order

print("Iterative DFS:", dfs_iterative(tree, root, "Nvidia"))
print("Recursive DFS:", dfs_recursive(tree, root, "Nvidia"))

### BFS versus DFS — interview comparison

| Question | BFS | DFS |
|---|---|---|
| Main structure | Queue (`deque`) | Stack / recursion |
| Exploration | Level by level | Branch by branch |
| Minimum number of edges to target? | **Yes**, if unweighted | **Not guaranteed** |
| Time, all nodes | O(N) | O(N) |
| Extra memory, typical tree | O(max width) | O(max depth) recursion/stack (plus pending siblings for iterative DFS) |
| Best example | Nearest relevant folder/document | Traverse all descendants, nested structure |

For iterative DFS on a general branching tree, pending siblings can increase memory beyond the exact depth; worst-case stack space is O(N). Recursive DFS in Python may hit the recursion limit for very deep trees.

## 4. How a search returns the *path* (not just True/False)
Interviewers often change “does target exist?” to “show me how to get there.” In BFS, carry a `(node, path)` pair or store parent pointers. The first target popped by BFS gives a **shortest path by edge count**.

In [ ]:
def bfs_path(tree, start, target):
    queue = deque([(start, [start])])
    while queue:
        node, path = queue.popleft()
        if node == target:
            return path
        for child in tree.get(node, []):
            queue.append((child, path + [child]))
    return None

print(bfs_path(tree, root, "Nvidia"))
print(bfs_path(tree, root, "Unknown"))

**Knowledge check:** Why is this implementation convenient but not the most memory-efficient? Each queue item stores a **whole path list**. A `parent` dictionary stores one predecessor per visited node and reconstructs the final path only once.

In [ ]:
def bfs_path_parent(tree, start, target):
    queue = deque([start])
    parent = {start: None}
    while queue:
        node = queue.popleft()
        if node == target:
            path = []
            while node is not None:
                path.append(node)
                node = parent[node]
            return path[::-1]
        for child in tree.get(node, []):
            if child not in parent:
                parent[child] = node
                queue.append(child)
    return None

assert bfs_path_parent(tree, root, "Nvidia") == ["Documents", "Research", "Equities", "Nvidia"]
print(bfs_path_parent(tree, root, "Governance"))

## 5. Applied example: searching text in a hierarchy of research documents
This is a **realistic text-search problem**, not just looking up exact node names. Imagine a firm stores documents by research area. We want documents whose **text contains a query**, ignoring case.

**Important distinction:** The **tree** tells us *which documents to visit*; the **text-matching rule** tells us *whether a document is relevant*. BFS/DFS alone do not understand meaning; here we use substring matching. Real financial research search may use an inverted index, BM25, semantic embeddings, or hybrid retrieval instead.

In [ ]:
documents = {
    "Research": {"text": "Investment research library", "children": ["Company Notes", "Thematic Notes"]},
    "Company Notes": {"text": "Company coverage and fundamentals", "children": ["Acme Note", "Beta Note"]},
    "Thematic Notes": {"text": "Sector and sustainable finance topics", "children": ["Carbon Note", "Risk Note"]},
    "Acme Note": {"text": "Acme reports lower carbon emissions and new climate targets", "children": []},
    "Beta Note": {"text": "Beta faces an ESG governance controversy", "children": []},
    "Carbon Note": {"text": "Carbon emissions estimation from revenue and industry", "children": []},
    "Risk Note": {"text": "Climate transition risk and credit spreads", "children": []},
}

def search_documents_bfs(documents, start, keyword):
    queue = deque([(start, 0)])
    matches = []
    while queue:
        node, depth = queue.popleft()
        record = documents[node]
        if keyword.lower() in record["text"].lower():
            matches.append((node, depth, record["text"]))
        queue.extend((child, depth + 1) for child in record["children"])
    return matches

for hit in search_documents_bfs(documents, "Research", "carbon"):
    print(hit)

### What if we want only the first result?
- BFS finds the **shallowest matching node** first (not necessarily the **most semantically relevant**).
- DFS might find an item deep under `Company Notes` before examining a shallower match under `Thematic Notes`.
- If searching text across **millions of documents**, scanning every node is inefficient. Build a **search index** instead.

**Exercise 5A:** Search for `climate` and `governance`. Predict the matching node titles before running.

In [ ]:
for keyword in ["climate", "governance", "credit", "unknown"]:
    print(keyword, "=>", [name for name, depth, txt in search_documents_bfs(documents, "Research", keyword)])

## 6. The essential interview distinction: tree vs graph
A tree cannot have cycles or multiple parents. In a general **graph**, suppliers can link to multiple companies and circular relationships can exist.

In a graph, **always consider a `visited` set** to prevent infinite loops and duplicate processing. The following ESG supplier network is a *graph*, not a tree.

In [ ]:
supplier_graph = {
    "Bank": ["Supplier A", "Supplier B"],
    "Supplier A": ["Factory", "Bank"],
    "Supplier B": ["Factory"],
    "Factory": ["Supplier A", "Risk Alert"],
    "Risk Alert": []
}

def graph_bfs(graph, start, target):
    queue = deque([start])
    visited = {start}
    parent = {start: None}
    while queue:
        node = queue.popleft()
        if node == target:
            result = []
            while node is not None:
                result.append(node)
                node = parent[node]
            return result[::-1]
        for nxt in graph.get(node, []):
            if nxt not in visited:
                visited.add(nxt)  # Mark when ENQUEUING, not later
                parent[nxt] = node
                queue.append(nxt)
    return None

print(graph_bfs(supplier_graph, "Bank", "Risk Alert"))

### Why mark a node as visited when enqueueing?
If `Factory` is reachable from both suppliers, without early marking we might queue it **twice**. Marking when adding ensures it is enqueued only once.

For a graph with **V vertices and E edges**, BFS and DFS take **O(V + E)** time using adjacency lists (not O(V) if there are many edges).

## 7. Another kind of text-related tree: a trie (prefix tree)
A **trie** stores strings one character per edge. A path from root to a marked end node spells a word. This is useful for **autocomplete, prefix lookup, dictionaries, ticker suggestions**.

Example words `car`, `carbon`, `cat`:
```text
(root)
   └── c
       └── a
           ├── r* ── b ── o ── n*
           └── t*
```
`*` indicates a complete word. The key idea: shared prefixes occupy shared nodes.

In [ ]:
END = "__end__"

def trie_insert(trie, word):
    node = trie
    for letter in word:
        node = node.setdefault(letter, {})
    node[END] = True

def trie_prefix_matches(trie, prefix):
    node = trie
    for letter in prefix:
        if letter not in node:
            return []
        node = node[letter]
    results = []
    def collect(node, built):
        if END in node:
            results.append(built)
        for letter, child in node.items():
            if letter != END:
                collect(child, built + letter)
    collect(node, prefix)
    return sorted(results)

trie = {}
for term in ["carbon", "car", "cat", "climate", "credit", "controversy"]:
    trie_insert(trie, term)
print("ca ->", trie_prefix_matches(trie, "ca"))
print("cr ->", trie_prefix_matches(trie, "cr"))
print("z  ->", trie_prefix_matches(trie, "z"))

**Trie vs scanning words:** If the prefix contains `L` characters, reaching its trie node costs O(L); collecting suggestions requires additional work proportional to matching output. Constructing a trie consumes additional memory. Tries are not automatically the best solution for *full-text* keyword search or semantic similarity.

## 8. A special case: binary search tree (BST)
A **binary tree** has at most two children per node. A **binary search tree** additionally obeys:

- Values in left subtree are smaller than the node.
- Values in right subtree are larger than the node.

```text
            8
          /   \
         3     10
        / \      \
       1   6      14
          / \     /
         4   7   13
```

Because it is ordered, you **do not need to traverse every node** to search for a key. Compare and choose left/right. In a *balanced* BST, search is O(log N); in a skewed BST, O(N). Ordinary document trees are not BSTs.

In [ ]:
class BSTNode:
    def __init__(self, value, left=None, right=None):
        self.value, self.left, self.right = value, left, right

bst = BSTNode(8,
    BSTNode(3, BSTNode(1), BSTNode(6, BSTNode(4), BSTNode(7))),
    BSTNode(10, None, BSTNode(14, BSTNode(13))))

def bst_search(node, target):
    comparisons = []
    while node is not None:
        comparisons.append(node.value)
        if target == node.value:
            return True, comparisons
        node = node.left if target < node.value else node.right
    return False, comparisons

for v in [7, 13, 12]:
    print(v, bst_search(bst, v))

## 9. Retrieval interview concept: tree traversal vs text search vs RAG
Interviewers may ask: “How do you find relevant disclosures?” Distinguish **three different problems**:

1. **Tree traversal:** Explore a hierarchical structure (directories, taxonomy, parse tree) via BFS/DFS.
2. **Lexical search:** Match words in document text. Production systems typically use an **inverted index** and ranking (e.g. BM25), not BFS over all documents.
3. **Semantic search / RAG:** Encode passages as embeddings, retrieve semantically related candidates (perhaps using an ANN vector index), rerank, then pass grounded passages to a language model. A hierarchical document tree can help scope the retrieval but does not substitute for a retrieval index.

**Bloomberg-style example:** Find references to *Scope 3 emissions* across company disclosures. Traversal can collect all disclosure files; a lexical or hybrid index can retrieve relevant passages, then an extraction model produces structured fields with provenance.

## 10. Guided coding exercises — do not immediately scroll to solutions
Fill in each `TODO`. Tests show expected behaviour. Start with the easiest and progress.

### Exercise A (easy): Count all nodes below the root
Write a recursive function `count_nodes(tree, node)` including the node itself. Expected result for `Documents`: 9.

In [ ]:
def count_nodes_your_turn(tree, node):
    # TODO: 1 for this node plus the count in every child's subtree
    raise NotImplementedError("Implement Exercise A")

# Once completed, try:
# assert count_nodes_your_turn(tree, "Documents") == 9

### Exercise B (easy): Maximum depth
Return the number of edges along the longest root → leaf path. For our tree the answer is **3**. Base case: what is the depth of a leaf?

In [ ]:
def max_depth_your_turn(tree, node):
    # TODO
    raise NotImplementedError("Implement Exercise B")

# assert max_depth_your_turn(tree, "Documents") == 3

### Exercise C (medium): Find the first matching text document with BFS
Accept a `keyword` and return `(node, depth)` for the **first** match. Return `None` otherwise. Use `deque`. Predict which result you will get for `climate`.

In [ ]:
def first_text_match_your_turn(documents, root, keyword):
    # TODO
    raise NotImplementedError("Implement Exercise C")

# assert first_text_match_your_turn(documents, "Research", "climate") == ("Acme Note", 2)

### Exercise D (medium): DFS without recursion
Return an ordered list of *all visited nodes*. Leftmost children must be visited first. Do not use recursion.

In [ ]:
def dfs_order_your_turn(tree, start):
    # TODO
    raise NotImplementedError("Implement Exercise D")

# assert dfs_order_your_turn(tree, root)[0:4] == ["Documents", "Research", "Equities", "Apple"]

### Exercise E (medium): Find the shortest chain of supplier relationships
Reuse `supplier_graph`. Return shortest path from `Bank` to `Risk Alert`; guard cycles using a visited set. Ask yourself: why BFS and not plain DFS?

In [ ]:
def shortest_supplier_path_your_turn(graph, start, goal):
    # TODO
    raise NotImplementedError("Implement Exercise E")

# assert shortest_supplier_path_your_turn(supplier_graph, "Bank", "Risk Alert") is not None

### Exercise F (stretch): Search a trie for complete words
Implement `trie_contains(trie, word)`; distinguish **prefix exists** from **whole word exists**. Example: `car` true, `carb` false, `carbon` true.

In [ ]:
def trie_contains_your_turn(trie, word):
    # TODO
    raise NotImplementedError("Implement Exercise F")

# assert trie_contains_your_turn(trie, "car") is True
# assert trie_contains_your_turn(trie, "carb") is False

## 11. Worked solutions and explanations
**Spoiler section.** Compare your reasoning to the implementations below, then rewrite them from memory.

In [ ]:
def count_nodes(tree, node):
    return 1 + sum(count_nodes(tree, child) for child in tree.get(node, []))

def max_depth(tree, node):
    children = tree.get(node, [])
    if not children:
        return 0                        # zero edges below a leaf
    return 1 + max(max_depth(tree, child) for child in children)

print("Number of nodes:", count_nodes(tree, root))
print("Maximum depth:", max_depth(tree, root))
assert count_nodes(tree, root) == 9
assert max_depth(tree, root) == 3

In [ ]:
def first_text_match(documents, root, keyword):
    q = deque([(root, 0)])
    while q:
        node, depth = q.popleft()
        if keyword.casefold() in documents[node]["text"].casefold():
            return node, depth
        q.extend((child, depth + 1) for child in documents[node]["children"])
    return None

def dfs_order(tree, start):
    stack = [start]
    order = []
    while stack:
        node = stack.pop()
        order.append(node)
        stack.extend(reversed(tree.get(node, [])))
    return order

print(first_text_match(documents, "Research", "climate"))
print(dfs_order(tree, root))
assert first_text_match(documents, "Research", "climate") == ("Acme Note", 2)
assert dfs_order(tree, root)[:4] == ["Documents", "Research", "Equities", "Apple"]

In [ ]:
def shortest_supplier_path(graph, start, goal):
    q = deque([start])
    parent = {start: None}                   # Acts as visited set as well
    while q:
        node = q.popleft()
        if node == goal:
            path = []
            while node is not None:
                path.append(node)
                node = parent[node]
            return path[::-1]
        for neighbor in graph.get(node, []):
            if neighbor not in parent:
                parent[neighbor] = node
                q.append(neighbor)
    return None

def trie_contains(trie, word):
    node = trie
    for letter in word:
        if letter not in node:
            return False
        node = node[letter]
    return END in node                     # Prefix != complete word

assert shortest_supplier_path(supplier_graph, "Bank", "Risk Alert") == ["Bank", "Supplier A", "Factory", "Risk Alert"]
assert trie_contains(trie, "car")
assert not trie_contains(trie, "carb")
assert trie_contains(trie, "carbon")
print("All solutions passed!")

## 12. Interview knowledge questions (answer aloud first)

1. Why does BFS need a queue, not a stack?
2. If you reverse the order of children added to an iterative DFS stack, what changes?
3. What does **shortest** mean when we say BFS finds a shortest path?
4. What breaks if a graph with cycles is searched with no `visited` set?
5. Describe time/space complexity of BFS on a tree and BFS on a graph.
6. Why does recursive DFS sometimes fail for very deep data?
7. What is the distinction between traversing a document hierarchy and querying full-text documents?
8. How is a trie different from a binary search tree?
9. How would you search for *the idea* of carbon footprint if the exact words do not appear?
10. How would you unit-test that BFS returns a shortest path?

### Suggested answers
1. Queue processes earlier-added nodes first, ensuring all depth-`d` nodes are handled before depth-`d+1`.
2. Traversal order changes; stack LIFO means the last child pushed is visited first.
3. Fewest **edges** (unweighted graph). For different edge costs use another algorithm, e.g. Dijkstra for nonnegative weights.
4. Duplicate work or an infinite loop; visited prevents revisiting nodes.
5. Tree O(N) time; queue up to O(N) / often O(width). Graph O(V+E) time, O(V) auxiliary space.
6. Python recursion depth limit and call-stack usage; switch to an explicit stack.
7. Traversal visits folders/nodes; full-text retrieval relies on indexing and ranking for efficiency.
8. Trie indexes characters and shared prefixes; BST compares ordered key values.
9. Semantic embeddings / hybrid search, with evaluation and document evidence.
10. Use a graph where DFS discovers a longer route but BFS finds a shorter one; assert returned number of edges.

## 13. Mini mock interview (30 minutes)
**Part I (5 min):** Draw the tree and recite BFS/DFS visit orders. Explain queue versus stack.

**Part II (10 min):** Without looking, implement `bfs_path_parent` and test an absent target.

**Part III (10 min):** Implement a case-insensitive document text search that returns all matching titles; discuss why it will not scale to 10 million disclosures.

**Part IV (5 min):** Explain how hierarchical navigation, lexical search, and embedding retrieval could be combined for sustainability controversies.

**Success criteria:** Correct algorithm, sensible variable names, no unnecessary O(N²) list queue operations, explanation of complexity, and at least 2 edge-case tests.

## 14. Common mistakes and how to debug them
- **BFS accidentally uses `pop()`** → becomes stack-like DFS. Use `popleft()`.
- **Iterative DFS children appear reversed** → `stack.extend(reversed(children))` to preserve left-to-right visitation.
- **Cycle causes infinite search** → add a `visited` set or `parent` dictionary.
- **Recursive function forgets `return`** → results from child calls are discarded.
- **Confuses tree search with BST search** → only prune by `<` or `>` when there is an ordering invariant.
- **Says BFS is semantically better for text** → BFS prioritises depth, not relevance.
- **No edge cases** → always test missing target, root target, leaf node, empty child lists, and graphs with cycles.

**Final mental shortcut:** **BFS = QUEUE = levels = fewest edges. DFS = STACK = deep first = backtracking.**

## 15. Extra practice: five prompts (no solutions supplied)
1. Return all paths from the root to leaves as lists of node names.
2. Given a document tree, return matches *only* under `Thematic Notes`.
3. Modify BFS to accept a predicate function, e.g. `lambda text: "carbon" in text.lower()`.
4. For a graph where edges are weighted by supplier risk, explain why ordinary BFS may not minimise total risk.
5. Build a tiny inverted index mapping each lowercased word to the document titles containing it. Compare lookup behaviour to scanning the tree.

If these feel difficult, revisit sections 2–5 and rerun the traces manually. The objective is to build intuition before speed.

---
# Part II — Text Manipulation & String Algorithms: Beginner → Advanced

**Goal:** become confident with string questions in coding interviews and understand how these tools complement tree/trie search. Examples use company names, financial headlines and ESG news.

**How to learn:** (1) predict the output, (2) run the cell, (3) change the inputs, (4) solve the exercises **before** reading the solutions. Complexity is shown in terms of `n` = length of text and `m` = length of pattern.

**Dependency:** this new part uses only Python's standard library, so it runs without installing packages.

## 16. String fundamentals: characters, slicing, immutability

A Python string is an **immutable** sequence of Unicode characters. You cannot change one character in place. `s[a:b]` starts at index `a`, stops **before** `b`. Negative indices count from the end.

Useful built-ins: `.lower()`, `.casefold()`, `.strip()`, `.split()`, `' '.join(...)`, `.replace()`, `.find()`, `.startswith()`, `enumerate()`.

- Membership (`needle in text`) asks whether a substring exists; it is *not* a tree search.
- `casefold()` is often better than `lower()` for Unicode-insensitive comparison.
- Repeated string `+=` in a loop can cause quadratic copying. Accumulate pieces and `''.join(...)`.

In [ ]:
headline = "  Bloomberg: GREEN Energy Outlook  "
clean = headline.strip().casefold()
print("Clean:", clean)
print("Prefix:", clean[:9], "Last word:", clean.split()[-1])
print("Words:", clean.split())
print("Contains energy?", "energy" in clean)
print("Reversed characters:", clean[::-1])
print("Rebuilt:", " | ".join(["ESG", "research", "news"]))
# Strings are immutable: use slicing to construct a modified copy.
example = "risk"
print(example[:1] + "a" + example[2:])

### Knowledge check 1

1. What does `s[1:4]` return for `s="carbon"`?
2. Why does `split()` usually differ from `split(" ")` on multiple spaces?
3. What is the difference between a substring, a subsequence, and a prefix?
4. Why should we **not** remove punctuation indiscriminately in finance text such as `S&P 500`, `$10.50` or `CO₂`?

## 17. Normalisation, tokenisation and word frequencies

A **tokeniser** divides text into meaningful pieces. A simple regex word tokeniser is sufficient for interview exercises, but production financial NLP needs specialised rules for tickers (`BRK.B`), currencies, dates, and multilingual content.

**Algorithm:** normalise → tokenise → count. `Counter` counts tokens in O(n) average time for n tokens, using hash-table operations.

In [ ]:
import re
from collections import Counter, defaultdict, deque

def simple_tokens(text):
    """Unicode word tokens; deliberately simple, not a production finance tokenizer."""
    return re.findall(r"\b\w+\b", text.casefold())

news = "Carbon risk is rising. Carbon disclosures help assess risk; carbon matters."
tokens = simple_tokens(news)
counts = Counter(tokens)
print(tokens)
print("Top 3:", counts.most_common(3))
print("Unique tokens:", sorted(set(tokens)))
assert counts["carbon"] == 3

## 18. Two pointers: palindrome with ignored punctuation

Two pointers start at opposite ends. Move inward while skipping non-alphanumeric characters. This avoids creating a fully cleaned copy.

**Time:** O(n); **additional space:** O(1). Unicode `isalnum` considers more than English letters.

In [ ]:
def is_palindrome(text):
    left, right = 0, len(text) - 1
    while left < right:
        if not text[left].isalnum():
            left += 1
        elif not text[right].isalnum():
            right -= 1
        elif text[left].casefold() != text[right].casefold():
            return False
        else:
            left += 1
            right -= 1
    return True

for value in ["A man, a plan, a canal: Panama!", "ESG", "", "Never odd or even"]:
    print(repr(value), is_palindrome(value))
assert is_palindrome("A man, a plan, a canal: Panama!")
assert not is_palindrome("ESG")

## 19. Hash maps: anagrams and character frequencies

Anagrams contain the same characters with the same multiplicities. Sorting both strings takes O(n log n); counters take O(n) expected time and O(k) space for k distinct characters. Decide upfront whether spaces/case/punctuation count.

In [ ]:
def are_anagrams(a, b):
    normalise = lambda s: [c for c in s.casefold() if c.isalnum()]
    return Counter(normalise(a)) == Counter(normalise(b))

print(are_anagrams("Debit card", "Bad credit"))
print(are_anagrams("carbon", "carbonn"))
assert are_anagrams("Debit card", "Bad credit")
assert not are_anagrams("carbon", "carbonn")

## 20. Sliding window: longest substring without repeated characters

**Substring** means adjacent characters. We maintain an active window `[left, right]` and the last position where each character occurred. When a repeated character is inside the current window, move `left` *past* its previous position.

Each pointer moves forward: **O(n) time**, O(k) space for distinct characters. Compare this with a naive O(n²) scan of all starting positions.

In [ ]:
def longest_unique_substring(s):
    last_seen = {}
    left, best_start, best_len = 0, 0, 0
    for right, char in enumerate(s):
        if char in last_seen and last_seen[char] >= left:
            left = last_seen[char] + 1
        last_seen[char] = right
        if right - left + 1 > best_len:
            best_start, best_len = left, right - left + 1
    return s[best_start:best_start + best_len]

for s in ["abcabcbb", "bbbbb", "pwwkew", "", "ESG-RISK"]:
    print(f"{s!r} -> {longest_unique_substring(s)!r}")
assert len(longest_unique_substring("pwwkew")) == 3

## 21. Sliding window: find all anagram occurrences

Find starting positions of all contiguous windows in `text` that are anagrams of `pattern`. For interview clarity, we use counts and update them as the window slides. **Time O(n + m)** on average (fixed-size alphabet comparison is O(1), while general Unicode dictionary comparison can depend on distinct symbols).

In [ ]:
def find_anagram_starts(text, pattern):
    if not pattern or len(pattern) > len(text):
        return []
    need = Counter(pattern)
    window = Counter(text[:len(pattern)])
    answer = []
    k = len(pattern)
    if window == need:
        answer.append(0)
    for i in range(k, len(text)):
        incoming, outgoing = text[i], text[i-k]
        window[incoming] += 1
        window[outgoing] -= 1
        if window[outgoing] == 0:
            del window[outgoing]
        if window == need:
            answer.append(i-k+1)
    return answer

print(find_anagram_starts("cbaebabacd", "abc"))
assert find_anagram_starts("cbaebabacd", "abc") == [0, 6]

## 22. Parsing text safely: extracting finance mentions

Regex (`re`) is useful for **syntactic** extraction (dates, labels, ticker-looking strings), but *not* enough to establish an entity identity or factual ESG controversy.

Below, use a deliberately narrow pattern to extract percentages. Never directly convert arbitrary extracted strings into trusted financial facts without validation.

In [ ]:
def extract_percentages(text):
    matches = re.finditer(r"(?<![\w.])([+-]?(?:\d+(?:\.\d+)?|\.\d+))\s*%", text)
    return [float(match.group(1)) for match in matches]

print(extract_percentages("Emissions fell 12.5%, but revenue grew +4%."))
assert extract_percentages("Target -3.0% then +1%") == [-3.0, 1.0]

## 23. Stack algorithm: validating nested brackets

This is a classic parsing interview question. Use a **stack** (last-in, first-out) because each closing bracket must match the latest unmatched opening bracket. **O(n) time**, O(n) worst-case space.

In [ ]:
def balanced_brackets(text):
    opening = set("([{ ".replace(" ", ""))
    matches = {")": "(", "]": "[", "}": "{"}
    stack = []
    for char in text:
        if char in opening:
            stack.append(char)
        elif char in matches:
            if not stack or stack.pop() != matches[char]:
                return False
    return not stack

for value in ["(ESG [risk])", "([)]", "{carbon: (scope1 + scope2)}", "(()"]:
    print(value, balanced_brackets(value))
assert balanced_brackets("{[()]}")
assert not balanced_brackets("([)]")

## 24. Pattern matching, from naive search to KMP

**Naive algorithm:** try matching a pattern at every starting position (worst-case O(nm)). Python's `str.find()` is excellent for production use, but an interview may ask you to implement the algorithm.

**KMP (Knuth–Morris–Pratt):** precompute the *longest proper prefix that is also a suffix* (`lps`) for each pattern prefix. On a mismatch, reuse existing matching information rather than restarting. Building `lps` costs O(m); scanning costs O(n). Total **O(n+m)** and O(m) extra space.

Example: pattern `ABABC` has LPS `[0,0,1,2,0]`.

In [ ]:
def prefix_table(pattern):
    lps = [0] * len(pattern)
    length = 0
    i = 1
    while i < len(pattern):
        if pattern[i] == pattern[length]:
            length += 1
            lps[i] = length
            i += 1
        elif length:
            length = lps[length-1]
        else:
            i += 1
    return lps

def kmp_find_all(text, pattern):
    if pattern == "":
        return list(range(len(text)+1))
    lps = prefix_table(pattern)
    results = []
    i = j = 0
    while i < len(text):
        if text[i] == pattern[j]:
            i += 1
            j += 1
            if j == len(pattern):
                results.append(i-j)
                j = lps[j-1]  # allow overlapping occurrences
        elif j:
            j = lps[j-1]
        else:
            i += 1
    return results

print("LPS:", prefix_table("ABABC"))
print("Occurrences:", kmp_find_all("ABABABCABABC", "ABABC"))
assert kmp_find_all("aaaa", "aa") == [0,1,2]
assert kmp_find_all("ESG", "xyz") == []

## 25. Edit distance: comparing noisy entity names

Levenshtein distance is the minimum number of **insertions, deletions and substitutions** needed to transform one string into another. Dynamic programming compares progressively longer prefixes. Recurrence:

- If final characters match: `dp[i][j] = dp[i-1][j-1]`.
- Otherwise: `1 + min(delete, insert, replace)`.

**Time O(nm)**, **space O(min(n,m))** with two rows. Useful for *candidate generation*, not final entity resolution: similar company names need IDs/context to disambiguate.

In [ ]:
def edit_distance(a, b):
    if len(a) < len(b):
        a, b = b, a  # b is the shorter string
    prev = list(range(len(b)+1))
    for i, ca in enumerate(a, start=1):
        curr = [i]
        for j, cb in enumerate(b, start=1):
            curr.append(min(
                prev[j] + 1,       # delete
                curr[j-1] + 1,     # insert
                prev[j-1] + (ca != cb)  # substitute/match
            ))
        prev = curr
    return prev[-1]

print("kitten → sitting:", edit_distance("kitten", "sitting"))
print("carbon → carbn:", edit_distance("carbon", "carbn"))
assert edit_distance("kitten", "sitting") == 3
assert edit_distance("", "ESG") == 3

## 26. Dynamic programming: longest common subsequence (LCS)

A **subsequence** preserves relative order but characters need not be adjacent. Example: `ACE` is a subsequence of `ABCDE`, but not a substring.

`dp[i][j]` = LCS length of prefixes `a[:i]`, `b[:j]`.

If last characters match, add one to diagonal; otherwise take max from above or left. **O(nm) time and O(nm) space** here, since we also reconstruct the actual subsequence.

In [ ]:
def longest_common_subsequence(a, b):
    n, m = len(a), len(b)
    dp = [[0]*(m+1) for _ in range(n+1)]
    for i in range(1, n+1):
        for j in range(1, m+1):
            dp[i][j] = (dp[i-1][j-1] + 1 if a[i-1] == b[j-1]
                        else max(dp[i-1][j], dp[i][j-1]))
    result = []
    i, j = n, m
    while i and j:
        if a[i-1] == b[j-1]:
            result.append(a[i-1]); i -= 1; j -= 1
        elif dp[i-1][j] >= dp[i][j-1]:
            i -= 1
        else:
            j -= 1
    return "".join(reversed(result))

print(longest_common_subsequence("ABCDE", "ACE"))
assert longest_common_subsequence("ABCDE", "ACE") == "ACE"

## 27. Search engine basics: inverted index and Boolean AND search

A trie searches **prefixes**; an inverted index maps each *token* to the documents containing it. Querying `carbon AND risk` intersects two postings sets. This is a simplified **lexical retrieval** building block and complements the document tree explored in Part I.

- Index building: O(total tokens) expected.
- Query intersections: proportional to size of postings sets (expected hash-set performance).
- Limitations: synonyms, morphological variants, ranking, and entity ambiguity.

In [ ]:
documents = {
    "d1": "Carbon risk disclosures highlight transition risk.",
    "d2": "Biodiversity controversies affect supply chain finance.",
    "d3": "Carbon emissions estimation can inform risk models.",
    "d4": "Materiality assessment combines climate and financial exposure.",
}

def build_inverted_index(docs):
    index = defaultdict(set)
    for doc_id, content in docs.items():
        for token in set(simple_tokens(content)):
            index[token].add(doc_id)
    return dict(index)

def and_search(index, query):
    terms = simple_tokens(query)
    if not terms:
        return []
    return sorted(set.intersection(*(index.get(term, set()) for term in terms)))

index = build_inverted_index(documents)
print("carbon AND risk ->", and_search(index, "carbon risk"))
print("biodiversity AND finance ->", and_search(index, "biodiversity finance"))
assert and_search(index, "carbon risk") == ["d1", "d3"]

## 28. Ranking text: TF-IDF / cosine similarity (without external libraries)

An inverted index finds matching documents. A retrieval system also needs **ranking**.

Here we implement a small **TF-IDF** model: TF is raw term count and IDF = `log((N+1)/(df+1))+1`; cosine similarity normalises for document length. This is a pedagogical baseline. BM25, neural embeddings, hybrid retrieval, metadata filters, and evaluation (e.g., Recall@k, nDCG) are common production extensions.

In [ ]:
import math

def tfidf_rank(docs, query):
    tokens_by_doc = {doc: simple_tokens(txt) for doc, txt in docs.items()}
    n = len(docs)
    df = Counter(t for toks in tokens_by_doc.values() for t in set(toks))
    idf = lambda t: math.log((n+1)/(df.get(t,0)+1)) + 1
    q_counts = Counter(simple_tokens(query))
    query_vec = {t: f*idf(t) for t, f in q_counts.items()}
    q_norm = math.sqrt(sum(v*v for v in query_vec.values()))
    scores = []
    for doc, toks in tokens_by_doc.items():
        d_vec = {t: f*idf(t) for t, f in Counter(toks).items()}
        d_norm = math.sqrt(sum(v*v for v in d_vec.values()))
        dot = sum(v*d_vec.get(t,0) for t,v in query_vec.items())
        scores.append((doc, dot/(q_norm*d_norm) if q_norm and d_norm else 0.0))
    return sorted(scores, key=lambda item: (-item[1], item[0]))

print(tfidf_rank(documents, "carbon emissions risk"))
assert tfidf_rank(documents, "carbon emissions")[0][0] == "d3"

## 29. Bringing it together: hierarchical document tree + lexical retrieval

**Question:** What happens when documents are stored in folders (`root → sector → company → article`)?

1. Traverse the **tree** using BFS to collect document nodes (structural search).
2. Build an inverted index over their text (lexical search).
3. Query terms and optionally rank with TF-IDF (content search).

In production, you normally index once and update incrementally rather than traverse the entire tree for every query.

In [ ]:
research_tree = {
    "name": "Research", "text": None, "children": [
        {"name": "Energy", "text": None, "children": [
            {"name": "ABC Energy", "text": "Carbon emissions and transition risk", "children": []}]},
        {"name": "Technology", "text": None, "children": [
            {"name": "XYZ Tech", "text": "Supply-chain controversy and labour governance", "children": []}]}
    ]
}

def collect_text_nodes_bfs(root):
    queue, result = deque([root]), {}
    while queue:
        node = queue.popleft()
        if node.get("text"):
            result[node["name"]] = node["text"]
        queue.extend(node.get("children", []))
    return result

leaf_docs = collect_text_nodes_bfs(research_tree)
print("Indexed documents:", leaf_docs)
print("carbon AND risk:", and_search(build_inverted_index(leaf_docs), "carbon risk"))
assert list(leaf_docs) == ["ABC Energy", "XYZ Tech"]

## 30. Additional advanced concepts (recognise them; don't memorise every implementation)

| Algorithm / data structure | Core idea | Typical use | Complexity |
|---|---|---|---|
| Trie | Character-by-character prefix tree | Autocomplete/ticker prefix lookup | O(L) lookup for prefix length L |
| KMP | Prefix/suffix reuse | Find exact pattern, overlapping hits | O(n+m) |
| Rabin–Karp | Rolling hash | Multiple candidate substring matches | O(n+m) expected, verify collisions |
| Aho–Corasick | Trie + failure links | Search many terms at once | O(text + matches + automaton) |
| Suffix array | Sorted suffix positions | Many substring queries | build depends on algorithm |
| Edit distance | Dynamic programming | Approximate name matching | O(nm) |
| LCS | Dynamic programming | Compare versions/sequences | O(nm) |
| Inverted index | Term → document postings | Keyword search | Query depends on postings sizes |
| BM25 | Term weighting + length normalisation | Ranked lexical retrieval | Depends on index/query |

**Interview judgement:** explain trade-offs and build simpler solutions correctly before choosing more advanced algorithms.

## 31. Exercises — attempt each before looking at solutions

### E1 (easy): Reverse words
Write `reverse_words(text)` such that `"  carbon   risk model " → "model risk carbon"`. Explain time/space complexity.

### E2 (easy): First non-repeating character
Write `first_unique_char(text)` returning the first single-occurrence character or `None`. Which Python collection will you use?

### E3 (medium): Longest substring with at most K distinct characters
Write `longest_k_distinct(s, k)` returning one longest contiguous substring. Test `"eceba", k=2 → "ece"`. Why is a sliding window appropriate?

### E4 (medium): Minimum window covering a pattern
Write `min_window(text, pattern)` returning the shortest substring containing all characters of `pattern` **including duplicates**. Test `"ADOBECODEBANC", "ABC" → "BANC"`.

### E5 (medium): Text search with AND / OR
Extend inverted-index searching with `or_search(index, query)`.

### E6 (medium): Find all occurrences with KMP
Predict indices for `kmp_find_all("aaaaa", "aaa")`; explain why overlap matters.

### E7 (advanced): Approximate company-name matching
Using `edit_distance`, return the closest candidate names to an input name, with distance and tie handling. Explain why this is insufficient for robust entity resolution.

### E8 (advanced, design): ESG controversy news alert
Sketch: ingest → clean/dedupe → entity recognition/linking → candidate retrieval → classification → temporal aggregation → alert and audit log. Discuss false positives, temporal leakage, precision/recall and drift.

**Interview drill:** Aim to write E1–E3 without hints, reason through E4, and explain E8 verbally.

In [ ]:
# EXERCISES: fill in the TODOs and remove `raise NotImplementedError`.
# Keep this cell as your independent practice area.

def exercise_reverse_words(text):
    raise NotImplementedError("TODO: implement reverse_words")

def exercise_first_unique_char(text):
    raise NotImplementedError("TODO: implement first_unique_char")

def exercise_longest_k_distinct(text, k):
    raise NotImplementedError("TODO: implement longest_k_distinct")

def exercise_min_window(text, pattern):
    raise NotImplementedError("TODO: implement min_window")

def exercise_or_search(index, query):
    raise NotImplementedError("TODO: implement OR retrieval")

## 32. Worked solutions — only read after attempting the exercises

The functions below have `solution_` prefixes so they do not overwrite your practice functions.

In [ ]:
def solution_reverse_words(text):
    return " ".join(reversed(text.split()))

def solution_first_unique_char(text):
    frequencies = Counter(text)
    return next((char for char in text if frequencies[char] == 1), None)

def solution_longest_k_distinct(text, k):
    if k <= 0:
        return ""
    count = defaultdict(int)
    left, best_start, best_len = 0, 0, 0
    for right, char in enumerate(text):
        count[char] += 1
        while len(count) > k:
            outgoing = text[left]
            count[outgoing] -= 1
            if count[outgoing] == 0:
                del count[outgoing]
            left += 1
        if right-left+1 > best_len:
            best_start, best_len = left, right-left+1
    return text[best_start:best_start+best_len]

def solution_min_window(text, pattern):
    if not pattern:
        return ""
    need = Counter(pattern)
    missing = len(pattern)
    left = 0
    best = (float("inf"), 0, 0)
    for right, char in enumerate(text):
        if need[char] > 0:
            missing -= 1
        need[char] -= 1
        while missing == 0:
            if right - left + 1 < best[0]:
                best = (right-left+1, left, right+1)
            old = text[left]
            need[old] += 1
            if need[old] > 0:
                missing += 1
            left += 1
    return text[best[1]:best[2]] if best[0] != float("inf") else ""

def solution_or_search(index, query):
    result = set()
    for term in simple_tokens(query):
        result.update(index.get(term, set()))
    return sorted(result)

def solution_closest_names(name, candidates):
    if not candidates:
        return []
    scored = [(candidate, edit_distance(name.casefold(), candidate.casefold())) for candidate in candidates]
    best = min(distance for _, distance in scored)
    return sorted([(candidate, dist) for candidate, dist in scored if dist == best])

assert solution_reverse_words("  carbon   risk model ") == "model risk carbon"
assert solution_first_unique_char("swiss") == "w"
assert solution_first_unique_char("aabb") is None
assert solution_longest_k_distinct("eceba", 2) == "ece"
assert solution_longest_k_distinct("abc", 0) == ""
assert solution_min_window("ADOBECODEBANC", "ABC") == "BANC"
assert solution_min_window("aa", "aa") == "aa"
assert solution_min_window("a", "aa") == ""
assert solution_or_search(index, "biodiversity carbon") == ["d1", "d2", "d3"]
assert kmp_find_all("aaaaa", "aaa") == [0,1,2]
print("All solution checks passed.")
print("Closest names:", solution_closest_names("Microsft", ["Microsoft", "Microchip", "Meta"]))

## 33. Self-assessment: active recall

Without executing cells, answer:

1. **BFS vs DFS:** which guarantees shortest path in an *unweighted* graph, and why?
2. **Stack vs queue:** which data structure underlies iterative DFS? BFS?
3. **Substring vs subsequence:** why are sliding windows appropriate for one but usually not the other?
4. **Counter vs set:** why does a set fail for anagram checking when characters repeat?
5. **KMP:** what does the prefix table save us from doing?
6. **Edit distance vs semantic similarity:** what does one measure that the other needn't?
7. **Trie vs inverted index:** prefix autocomplete vs retrieving articles about a topic?
8. **Financial NLP:** why should company name matching use entity IDs and dates, not names alone?
9. **Model evaluation:** what is more costly for an ESG controversy alert—false alarms or missed controversies? How would you quantify it?
10. **Complexity:** explain time and space for a two-pointer palindrome check, sliding window, BFS on a tree and Levenshtein distance.

**Suggested answer cues:** 1 BFS, 2 stack/queue, 3 contiguous vs noncontiguous, 4 multiplicity, 5 repeated comparisons, 6 lexical edits vs meaning, 7 trie/index, 8 ambiguity/renames, 9 application-specific thresholds, 10 O(n)/O(1), O(n)/O(k), O(V+E)/O(V), O(nm)/O(min(n,m)).

## 34. Final mock interview (45 minutes)

- **0–5 min:** Explain how tree traversal differs from searching inside a document.
- **5–15 min:** Implement `solution_longest_k_distinct` from scratch; walk through `eceba`, K=2.
- **15–25 min:** Implement first unique character and discuss time/space complexity.
- **25–35 min:** Design a searchable Bloomberg-style ESG document store with folder hierarchy, inverted index and reranking.
- **35–45 min:** Discuss monitoring: precision/recall, source bias, concept drift, entity linking, explainability and auditability.

**Progress milestone:** you can trace each algorithm by hand, implement core examples without copy/pasting, and explain *why* it is the right data structure.